<a href="https://colab.research.google.com/github/pepealania/agentic-rag/blob/main/LangGraph2Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import sys
import langgraph
import importlib.metadata

print("Python:", sys.version)
print("LangGraph:", importlib.metadata.version('langgraph'))

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
LangGraph: 1.2.9


In [6]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


class State(TypedDict):
    consulta: str
    evidencias: list[str]
    resultado: str


def recuperar(state: State):
    print(">>> Nodo 1: recuperar")

    evidencias = [
        "Documento A: LangGraph permite construir workflows con nodos y aristas.",
        "Documento B: El estado puede compartirse entre los nodos."
    ]

    return {
        "evidencias": evidencias
    }


def analizar(state: State):
    print(">>> Nodo 2: analizar")

    resultado = (
        f"Consulta: {state['consulta']}\n"
        f"Evidencias: {len(state['evidencias'])}\n"
        f"Conclusión: las evidencias son suficientes."
    )

    return {
        "resultado": resultado
    }


# ------------------------------------------------------------
# Construcción
# ------------------------------------------------------------

builder = StateGraph(State)

builder.add_node("recuperar", recuperar)
builder.add_node("analizar", analizar)

builder.add_edge(START, "recuperar")
builder.add_edge("recuperar", "analizar")
builder.add_edge("analizar", END)


# ------------------------------------------------------------
# Checkpointer
# ------------------------------------------------------------

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# ------------------------------------------------------------
# Ejecución
# ------------------------------------------------------------

config = {
    "configurable": {
        "thread_id": "experimento-001"
    }
}

result = graph.invoke(
    {
        "consulta": "¿Qué framework permite controlar un workflow agentic?",
        "evidencias": [],
        "resultado": ""
    },
    config
)


print("\n========== RESULTADO ==========")
print(result["resultado"])


# ------------------------------------------------------------
# HISTORIAL DE EJECUCIÓN
# ------------------------------------------------------------

print("\n========== HISTORIAL ==========")

history = list(
    graph.get_state_history(config)
)

for i, snapshot in enumerate(history):

    print(f"\n--- Snapshot {i} ---")

    print("Step:", snapshot.metadata.get("step"))
    print("Next:", snapshot.next)
    print("Values:", snapshot.values)

>>> Nodo 1: recuperar
>>> Nodo 2: analizar

========== RESULTADO ==========
Consulta: ¿Qué framework permite controlar un workflow agentic?
Evidencias: 2
Conclusión: las evidencias son suficientes.

========== HISTORIAL ==========

--- Snapshot 0 ---
Step: 2
Next: ()
Values: {'consulta': '¿Qué framework permite controlar un workflow agentic?', 'evidencias': ['Documento A: LangGraph permite construir workflows con nodos y aristas.', 'Documento B: El estado puede compartirse entre los nodos.'], 'resultado': 'Consulta: ¿Qué framework permite controlar un workflow agentic?\nEvidencias: 2\nConclusión: las evidencias son suficientes.'}

--- Snapshot 1 ---
Step: 1
Next: ('analizar',)
Values: {'consulta': '¿Qué framework permite controlar un workflow agentic?', 'evidencias': ['Documento A: LangGraph permite construir workflows con nodos y aristas.', 'Documento B: El estado puede compartirse entre los nodos.'], 'resultado': ''}

--- Snapshot 2 ---
Step: 0
Next: ('recuperar',)
Values: {'consulta